In [11]:
from ultralytics import YOLO
from manga_ocr import MangaOcr
import cv2
from PIL import Image
from transformers import pipeline
import numpy as np

In [12]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

In [13]:
panel_detector = YOLO("models/mosesb.pt")
ocr = MangaOcr()
bubble_detector = YOLO("models/comic-speech-bubble-detector.pt")

2026-02-23 15:22:17.327 | INFO     | manga_ocr.ocr:__init__:16 - Loading OCR model from kha-white/manga-ocr-base
Loading weights: 100%|██████████| 264/264 [00:00<00:00, 400.55it/s, Materializing param=encoder.pooler.dense.weight]                                       
The tied weights mapping and config for this model specifies to tie decoder.bert.embeddings.word_embeddings.weight to decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie decoder.cls.predictions.bias to decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MangaOcrModel LOAD REPORT from: kha-white/manga-ocr-base
Key                                  | Status     |  | 
------------------

In [14]:
# ── Device & Quantization Setup (adjust based on your GPU) ─────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

model = M2M100ForConditionalGeneration.from_pretrained("models/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("models/m2m100_418M")
# print("M2M100 languages:", tokenizer.lang_code_to_id.keys())

Loading weights: 100%|██████████| 512/512 [00:01<00:00, 434.27it/s, Materializing param=model.shared.weight]                                   
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [15]:
image_file = "img/008.jpg"
img = cv2.imread(image_file)

In [16]:
panel_results = panel_detector(img, conf=0.2)
anotation = panel_results[0].plot()
img_results ="panel_detection.jpg"
cv2.imwrite(img_results,anotation)


0: 640x480 6 Comic Panels, 1529.8ms
Speed: 39.3ms preprocess, 1529.8ms inference, 26.0ms postprocess per image at shape (1, 3, 640, 480)


True

number = 1
panels = []
for box in panel_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    panels.append({
        "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        "crop": img[y1:y2, x1:x2]
    }
    )
    crop = img[y1:y2, x1:x2]
    cv2.imwrite(f"crop{number}.jpg",crop)
    number +=1

In [17]:
h, w = img.shape[:2]

# ── Helper: Compute IoU between two boxes [x1,y1,x2,y2] ───────────────────────
def compute_iou(boxA, boxB):
    # Intersection coordinates
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    
    inter_area = max(0, xB - xA) * max(0, yB - yA)
    
    # Areas
    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    if boxA_area + boxB_area - inter_area == 0:
        return 0.0
    return inter_area / float(boxA_area + boxB_area - inter_area)

# ── Step 1: Detect panels on full page ───────────────────────────────────────
print("Detecting panels...")
panel_results = panel_detector(img, conf=0.25)
panels = []
for box in panel_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf)
    if conf >= 0.25:
        panels.append({
            "box": [x1, y1, x2, y2],   # for IoU
            "crop": img[y1:y2, x1:x2],
            "conf": conf
        })
print(f"Found {len(panels)} panels")

# ── Step 2: Detect bubbles on FULL page ──────────────────────────────────────
print("Detecting bubbles on full page...")
bubble_results = bubble_detector(img, conf=0.3)
bubbles = []
for box in bubble_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf)
    if conf >= 0.25:
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        bubble_crop = img[y1:y2, x1:x2]
        bubble_pil = Image.fromarray(cv2.cvtColor(bubble_crop, cv2.COLOR_BGR2RGB))
        
        bubbles.append({
            "box": [x1, y1, x2, y2],
            "cx": cx,
            "cy": cy,
            "crop_pil": bubble_pil,
            "conf": conf
        })
print(f"Found {len(bubbles)} bubbles")

# ── Step 3: Assign bubbles to panels using best IoU ──────────────────────────
panel_to_bubbles = {i: [] for i in range(len(panels))}

for bubble in bubbles:
    best_iou = 0.0
    best_panel_idx = -1
    
    for idx, panel in enumerate(panels):
        iou = compute_iou(bubble["box"], panel["box"])
        if iou > best_iou:
            best_iou = iou
            best_panel_idx = idx
    
    # Fallback: if no good IoU, assign by center point
    if best_iou < 0.1:  # adjust threshold if needed
        bubble_center = (bubble["cx"], bubble["cy"])
        for idx, panel in enumerate(panels):
            bx1, by1, bx2, by2 = panel["box"]
            if bx1 <= bubble["cx"] <= bx2 and by1 <= bubble["cy"] <= by2:
                best_panel_idx = idx
                break
    
    if best_panel_idx != -1:
        panel_to_bubbles[best_panel_idx].append(bubble)
    else:
        print(f"Warning: Bubble {bubble['box']} not assigned to any panel")

# ── Step 4: Sort panels in reading order ─────────────────────────────────────
def get_center(item):
    box = item["box"] if "box" in item else [item["x1"], item["y1"], item["x2"], item["y2"]]
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)

# Sort: top-to-bottom (cy asc), then right-to-left (-cx desc)
sorted_panel_indices = sorted(
    range(len(panels)),
    key=lambda i: (get_center(panels[i])[1], -get_center(panels[i])[0])
)

print(f"Sorted {len(sorted_panel_indices)} panels in manga reading order")

# ── Step 5: For each sorted panel → sort its bubbles ─────────────────────────
all_ordered_bubbles = []

for panel_idx in sorted_panel_indices:
    panel_bubbles = panel_to_bubbles[panel_idx]
    
    # Sort bubbles inside panel: top-to-bottom, right-to-left
    sorted_bubbles = sorted(
        panel_bubbles,
        key=lambda b: (b["cy"], -b["cx"])
    )
    
    all_ordered_bubbles.extend(sorted_bubbles)

print(f"Final ordered bubbles: {len(all_ordered_bubbles)}")

Detecting panels...

0: 640x480 6 Comic Panels, 1619.2ms
Speed: 6.4ms preprocess, 1619.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 480)
Found 6 panels
Detecting bubbles on full page...

0: 1024x736 13 text_bubbles, 1154.1ms
Speed: 15.9ms preprocess, 1154.1ms inference, 2.9ms postprocess per image at shape (1, 3, 1024, 736)
Found 13 bubbles
Sorted 6 panels in manga reading order
Final ordered bubbles: 13


In [18]:
def translate_ja_to_ar_m2m100(japanese_text: str, max_length: int = 200) -> str:
    tokenizer.src_lang = "ja"      
    tokenizer.tgt_lang = "ar"
    inputs = tokenizer(japanese_text, return_tensors="pt")
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id("ar"),
        max_length=max_length,
        num_beams=5,
        early_stopping=True
    )
    translated = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0].strip()
    return translated

In [19]:
bubble_number = 1
for bubble in all_ordered_bubbles:
    japanese = ocr(bubble["crop_pil"]).strip()
    print(f"Bubble {bubble_number}:")
    print(f"Japanese: {japanese}")
    print(f"Franch :{translate_ja_to_ar_m2m100(japanese)}")
    print("─" * 50)
    bubble_number += 1

Bubble 1:
Japanese: 風船が割れてパラシュートで落ちてきて．．．
Franch :لقد سقطت البالونات، وسقطت في المظلة..
──────────────────────────────────────────────────
Bubble 2:
Japanese: カメラを回収したら地球の写真が撮れてるんです！
Franch :إذا قمت بتصوير الكاميرا ، فستقوم بتصوير الأرض!
──────────────────────────────────────────────────
Bubble 3:
Japanese: 以前テレビで見たんです！
Franch :لقد رأيته في التلفزيون من قبل!
──────────────────────────────────────────────────
Bubble 4:
Japanese: 風船でブワフワとカメラが宇宙へ行って．．．
Franch :على متن البالونات و الكاميرا في الفضاء..
──────────────────────────────────────────────────
Bubble 5:
Japanese: ね？すっごくワクワクしませんか？
Franch :هل أنت متحمس للغاية؟
──────────────────────────────────────────────────
Bubble 6:
Japanese: なっ
Franch :نعم نعم
──────────────────────────────────────────────────
Bubble 7:
Japanese: それに．．．
Franch :وبالإضافة إلى...
──────────────────────────────────────────────────
Bubble 8:
Japanese: だからたくさん風船がいるんですっ
Franch :وهناك الكثير من الباليه.
──────────────────────────────────────────────────
Bubble 9:
Japan